# Data Cleaning

Steps:
> Convert feet to meters

> Split dataset into 4 unique locations

> Group lanes by directionality

> Fidelity checking data quality assurance

In [ ]:
import os
import cudf
import pandas as pd
import matplotlib.pyplot as plt
import random

In [ ]:
def load_and_convert_units(path):
    """Load NGSIM CSV and convert units to SI (meters, m/s, m/s²)."""
    df = cudf.read_csv(path)

    ft_to_m = 0.3048
    df["Local_X_m"] = df["Local_X"] * ft_to_m
    df["Local_Y_m"] = df["Local_Y"] * ft_to_m
    df["v_Vel_mps"] = df["v_Vel"] * ft_to_m
    df["v_Acc_mps2"] = df["v_Acc"] * ft_to_m
    
    df["v_length_m"] = df["v_length"] * ft_to_m

    return df

In [ ]:
def split_by_location(df):
    """Reduce columns and split dataset into one per unique location."""
    df = df[[
        "Vehicle_ID", "Frame_ID", "Global_Time", "Local_X_m", "Local_Y_m",
        "v_Vel_mps", "v_Acc_mps2", "Lane_ID", "Preceding", "Following",
        "Space_Headway", "Time_Headway", "v_length_m", "Location"
    ]]

    locations = df["Location"].unique()
    subsets = []

    for location in locations.to_pandas():
        df_loc = df[df["Location"] == location]
        subsets.append((location, df_loc))

    return subsets

In [ ]:
def infer_direction_groups(df):
    print("Inferring lane directions using cuDF and Global_Time...")

    vehicle_movements = []

    # Group by Vehicle_ID
    for vid, group in df.groupby("Vehicle_ID"):
        group = group.dropna(subset=["Local_Y_m"])
        if len(group) < 2:
            continue

        group = group.sort_values("Global_Time")

        start_y = group["Local_Y_m"].iloc[0]
        end_y = group["Local_Y_m"].iloc[-1]
        lane_id = group["Lane_ID"].iloc[0]

        if start_y is not None and end_y is not None:
            direction = 1 if float(end_y - start_y) > 0 else -1
            vehicle_movements.append((int(lane_id), direction))

    # Convert to cuDF
    movement_df = cudf.DataFrame(vehicle_movements, columns=["Lane_ID", "Direction"])

    # Average direction per lane
    grouped = movement_df.groupby("Lane_ID").agg({"Direction": "mean"})
    grouped["Direction"] = grouped["Direction"].apply(lambda x: 1 if float(x) > 0 else -1)


    # Map -1 → 0, +1 → 1
    direction_map = {dir_val: i for i, dir_val in enumerate(sorted(grouped["Direction"].unique().to_pandas()))}
    grouped["Direction_Group"] = grouped["Direction"].replace(direction_map).astype("int8")
    grouped = grouped.drop(columns=["Direction"])

    # Merge back
    df = df.merge(grouped, on="Lane_ID", how="left")

    print(f"✅ Inferred direction groups: {df['Direction_Group'].unique().to_pandas().tolist()}")
    return df


In [ ]:
def remove_invalid_headways(df):
    """Remove rows where Space_Headway < preceding vehicle's length."""
    # Join to get preceding vehicle's v_length
    df_prec = df[["Vehicle_ID", "Global_Time", "v_length_m"]].rename(columns={
        "Vehicle_ID": "Preceding_ID",
        "v_length_m": "Preceding_v_Length"
    })

    df = df.merge(df_prec, left_on=["Preceding", "Global_Time"],
                         right_on=["Preceding_ID", "Global_Time"], how="left")

    # Filter out invalid rows
    df = df[df["Space_Headway"] >= df["Preceding_v_Length"]]

    return df

In [ ]:
def enforce_preceding_following_symmetry(df):
    """Ensure vehicle A's preceding is vehicle B, and B's following is A (same timestamp)."""
    df_a = df[["Vehicle_ID", "Global_Time", "Preceding"]]
    df_b = df[["Vehicle_ID", "Global_Time", "Following"]].rename(columns={
        "Vehicle_ID": "Preceding",
        "Following": "Reverse_Lookup_ID"
    })

    # Join A's Preceding with B's Following
    joined = df_a.merge(df_b, on=["Preceding", "Global_Time"], how="inner")

    # Keep only symmetric matches
    symmetric_ids = joined[joined["Vehicle_ID"] == joined["Reverse_Lookup_ID"]]["Vehicle_ID"].unique()

    # Keep only valid rows from original df
    df = df[df["Vehicle_ID"].isin(symmetric_ids)]

    return df

In [ ]:
def visualize(df):
    df_pd = df.to_pandas()

    # Group by Lane_ID
    lane_groups = dict(tuple(df_pd.groupby("Lane_ID")))

    # Create a plot
    plt.figure(figsize=(15, 6))

    for lane_id, lane_df in lane_groups.items():
        lane_df_sorted = lane_df.sort_values("Global_Time")
        vehicles = lane_df_sorted["Vehicle_ID"].unique()
        if len(vehicles) == 0:
            continue

        # Pick a random vehicle from the lane
        random_vehicle = random.choice(vehicles)
        vehicle_data = lane_df_sorted[lane_df_sorted["Vehicle_ID"] == random_vehicle]

        # Plot the vehicle's Local_Y_m over time
        plt.plot(vehicle_data["Global_Time"], vehicle_data["Local_Y_m"], label=f"Lane {lane_id} - Vehicle {random_vehicle}")

    plt.xlabel("Global Time (ms)")
    plt.ylabel("Local Y Position (m)")
    plt.title("Random Vehicle Trajectory per Lane")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.grid(True)
    plt.show()

In [ ]:
def clean_ngsim_all_locations(input_path):
    output_dir = "Cleaned_NGSIM"
    os.makedirs(output_dir, exist_ok=True)

    df = load_and_convert_units(input_path)
    print(f"Total loaded rows: {len(df)}")

    location_datasets = split_by_location(df)
    for location, df_loc in location_datasets:
        visualize(df_loc)
        print("\n\n\nDone")

    for location, df_loc in location_datasets:
        print(f"\n📍 Processing location: {location} ({len(df_loc)} rows)")
        df_loc = infer_direction_groups(df_loc)
        print(f"Direction groups: {df_loc['Direction_Group'].unique().to_pandas()}")

        df_loc = remove_invalid_headways(df_loc)
        print(f"After headway filtering: {len(df_loc)} rows")

        df_loc = enforce_preceding_following_symmetry(df_loc)
        print(f"After symmetry check: {len(df_loc)} rows")

        filename = f"{output_dir}/cleaned_ngsim_{location}.csv"
        df_loc.to_csv(filename, index=False)
        print(f"✅ Saved cleaned file: {filename}")

In [ ]:
clean_ngsim_all_locations("../NGSIM.csv")
